### Automotive

In [0]:
# 01 Imports

from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import pyspark.sql.functions as f

In [0]:
# 02 Config

PICTURE_VOLUME_PATH = "/Volumes/agentbricks/volumes/pictures"
METADATA_TABLE = "agentbricks.sector_data_bronze.report_picture_metadata"

APP_BASE_URL = "https://agent-picture-retrieving-app-3863256616093854.14.azure.databricksapps.com"

source_table = "agentbricks.sector_data_bronze.car_fleet_cz"

sector = "automotive"
chart_id = "personal_vehicles_registered_per_year"
chart_title = "Personal Vehicles Registered in Czechia by Year"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

chart_filename = f"{chart_id}_{timestamp}.png"
chart_path = f"{PICTURE_VOLUME_PATH}/{chart_filename}"
image_url = f"{APP_BASE_URL}/image/{chart_filename}"
markdown_reference = f"![{chart_title}]({image_url})"

In [0]:
# 03 Load source data

df_car_fleet_cz = spark.table(source_table)

df_chart = (
    df_car_fleet_cz
    .select(
        f.col("year"),
        f.col("oa_registred_number").alias("personal_vehicles_registered"),
    )
    .where(f.col("year").isNotNull())
    .where(f.col("personal_vehicles_registered").isNotNull())
    .orderBy("year")
)

display(df_chart)

In [0]:
# 04 Prepare chart data

pdf_chart = df_chart.toPandas()

first_year = int(pdf_chart["year"].min())
last_year = int(pdf_chart["year"].max())

first_value = int(
    pdf_chart.loc[
        pdf_chart["year"] == first_year,
        "personal_vehicles_registered",
    ].iloc[0]
)

last_value = int(
    pdf_chart.loc[
        pdf_chart["year"] == last_year,
        "personal_vehicles_registered",
    ].iloc[0]
)

absolute_change = last_value - first_value
percentage_change = absolute_change / first_value * 100

In [0]:
# 04 Prepare chart data

pdf_chart = df_chart.toPandas()

first_year = int(pdf_chart["year"].min())
last_year = int(pdf_chart["year"].max())

first_value = int(
    pdf_chart.loc[
        pdf_chart["year"] == first_year,
        "personal_vehicles_registered",
    ].iloc[0]
)

last_value = int(
    pdf_chart.loc[
        pdf_chart["year"] == last_year,
        "personal_vehicles_registered",
    ].iloc[0]
)

absolute_change = last_value - first_value
percentage_change = absolute_change / first_value * 100

In [0]:
# 05 Generate and save picture

plt.figure(figsize=(10, 6))

plt.plot(
    pdf_chart["year"],
    pdf_chart["personal_vehicles_registered"],
    marker="o",
)

plt.title(chart_title)
plt.xlabel("Year")
plt.ylabel("Number of Registered Personal Vehicles")
plt.grid(True)
plt.tight_layout()

plt.savefig(chart_path, dpi=150)
plt.close()

print(f"Chart saved to: {chart_path}")

In [0]:
# 06 Create metadata row

chart_description = (
    "The chart shows the yearly development of registered personal vehicles "
    "in Czechia based on the oa_registred_number column."
)

suggested_commentary = (
    f"The number of registered personal vehicles increased from {first_value:,} "
    f"in {first_year} to {last_value:,} in {last_year}, representing an increase "
    f"of {absolute_change:,} vehicles, or approximately {percentage_change:.1f}%."
)

metadata = [
    {
        "chart_id": chart_id,
        "sector": sector,
        "chart_title": chart_title,
        "chart_description": chart_description,
        "suggested_commentary": suggested_commentary,
        "image_filename": chart_filename,
        "image_path": chart_path,
        "image_url": image_url,
        "markdown_reference": markdown_reference,
        "source_table": source_table,
        "columns_used": "year, oa_registred_number",
        "created_at": datetime.now(),
    }
]

df_metadata = spark.createDataFrame(metadata)

display(df_metadata)

In [0]:
# 07 Save metadata to Delta table

(
    df_metadata
    .write
    .mode("append")
    .format("delta")
    .saveAsTable(METADATA_TABLE)
)

print(f"Metadata saved to table: {METADATA_TABLE}")